# A Walkthrough of the Image Analysis Code

## This notebook is an attempt to leave notes walking through the code and also share where some common errors might arise. The entritey of the code can be found on GitHub.

### If you have questions, email me: meckh776@live.kutztown.edu or text me: 484-538-5803

### GOOD LUCK!!!

# PIV CODE 
## A guide to using the PIV code for image Analysis 

### Masking Feature
In both the 'masked' and 'unmasked' versions of the code use an automask feature, sometimes you might have to manually create a secondary mask layer to isolate your region of interest.

This secondary manual masking feature will probably most useful for the compartment data to isolate a singular compartment for analysis. It can also be useful in the wave images where you have multiple 'bubbles' that are oscialling at different times. 

### Note about the PIV code for compartments

The code is not perfect and definitely need to be troubleshooted. You'll see in the GIFs that the largest event mechanically is depolyermization but the GIFs don't really capture this. I think this is a sensitivity issue to the images not being super clear when looking at a singular cluster. I feel confident that it is capturing movement but I am super confident in it's ability to capture movement on a smaller scale. USE WITH DISCRETION!!!

### Algorithmic Mask Version (no manual masking): 'Algo_maskPIV.py' 

First you will want to import these libraries. Note: all of this code is in python. I prefer to use the console **Anaconda** and the code runner **Spyder**, but you can use whatever you feel comfortable with. 

In [6]:
from glob import glob
import os
import matplotlib.pyplot as plt
import numpy as np
from openpiv import pyprocess, validation, filters, preprocess
from skimage.filters import gaussian, threshold_local
from skimage.util import img_as_float, img_as_ubyte
from skimage.filters import gaussian, threshold_otsu
from skimage.draw import polygon
import imageio.v3 as iio
from skimage.io import imread

ModuleNotFoundError: No module named 'openpiv'

**Note**: You might have to add some of these libraries via your termial. Depending on your computer type, the instructions will vary. 'imageio' library will probably thrown an error since it is not a common library to use. When you run this code in HERE, you will see the kind of error will most likley be thrown --> **"No module named 'openpiv'"**

Just google how to add this library to your console. 

Now, we will declare this variable, it initalize it so that later on we don't get errors. 

In [7]:
np.int = int

Now, we have to create the path inwhich we are pulling the images from. This is also something you can google if you get stuck. This is what mine looks like on my mac. Yours will most likley be different and if you are pulling directly from the server, **MAKE SURE YOU ARE CONNECTED TO THE SERVER!!!** otherwise, you will get an error.  

In [8]:
search_path = "/Volumes/qiongy-data/Users/Maggie/111725 compartment data/POS 3, all cycles2/Cycle 12-1/Pos3-1*.tif"
image_paths = sorted(glob(search_path))

To ensure that we are pulling the correct number of files, I created this print check. Make sure that this number matches the number of files you are looking at. 

In [9]:
if len(image_paths) < 2:
    raise ValueError(
        f"Found {len(image_paths)} images at path: {search_path}\n"
        "Double check your path!"
    )
print(f"Successfully tracked {len(image_paths)} frames for processing.")

ValueError: Found 0 images at path: /Volumes/qiongy-data/Users/Maggie/111725 compartment data/POS 3, all cycles2/Cycle 12-1/Pos3-1*.tif
Double check your path!

Now, we can finally start to process the data. Now the following code may not be compatible for non-mac computers so troubleshoot if needed.  

We are creating a temp file holding space to ensure that the computer can process the images without crashing. 

In [10]:
first_img = imread(image_paths[0])
#creating a temperary holding space for the files (wicked meme mention)
#this will help keep compuational expense lower
temp_dir = "temp_frames"
os.makedirs(temp_dir, exist_ok=True)

NameError: name 'imread' is not defined

The following code if for our figure output. 

You can change the **vmin** and **vmax** settings to get more aestheticlly pleasing results but make sure that if you are comparing two positions, both the vmin and vmax are the same for both.

You can also change your **scale** value as well to get a more favorable image. 

In [ ]:
vmin=0.0
vmax=5.0

#plotting the scale bar
fig, ax = plt.subplots(figsize=(8, 8))
dummy_Q = ax.quiver([0], [0], [0], [0], [0], cmap="plasma", scale=115, clim=(vmin,vmax))
cb = fig.colorbar(dummy_Q, ax=ax, orientation='horizontal', pad=0.08)
cb.set_label('Velocity Magnitude')
fig.tight_layout()

Next, we create **bins** to hold our data as we are processing the images.

In [ ]:
frame_files = []
total_pairs = len(image_paths) - 1

# Storage lists for overall analytics
time_list = []
avg_speed_list = []
max_speed_list = []
avg_move_x_list = []
avg_move_y_list = []

### The loop: 
This is where all of the image processing goes down. A lot of these functions were taken from openPIV so please look there if my explainations do not make full sense. 

The following block just breaks the process into chunks of 10 instead of loading all of the images in. This is to help not use up all of the RAM. 

In [ ]:
print(f"Processing {total_pairs} frame pairs...")

#LOOP WHERE BIG STUFF HAPPENS
for i in range(total_pairs):
    #counter for the images being processes
    if (i + 1) % 10 == 0 or i == 0 or i == total_pairs - 1:
        print(f"Processing frame {i + 1} of {total_pairs}...")

Next, since this code compares changes across two images, we must declare those two images. Then we nomalized the images as a float.

In [ ]:
    #layering the frames ontop of one another to measure pixel change
    curr_frame = imread(image_paths[i])
    next_frame = imread(image_paths[i + 1])
    
    #scaling our array to more standardized ranges (0-->1)
    curr_float = img_as_float(curr_frame)
    next_float = img_as_float(next_frame)

Now this step is really important!!! We are using a **gaussian** to blur our changes that fall below a certain **threshold**. That **threshold** is called **sigma**. This value will need to be changed at your descretion. 

After a few frames have loaded into the code, stop the code and see a sample output in your sandbox. See if you are picking up any undesirable noise and then alter your **sigma** value. 

The greater the **sigma** the less noise, and vice versa. Essentially, as you raise the sigma, you start to only see larger differnces between images.  

In [ ]:
    #blurring inbetween bc we just need to see movement in general, inc sigma for more blur
    curr_denoised = gaussian(curr_float, sigma=1.7)
    next_denoised = gaussian(next_float, sigma=1.7)

Next, we set this threshold to the local region of analysis and generalize it for each frame. We use the **Otsu** threshold since it has yeilded the best results so far. 

In [ ]:
    #changes threshold value per round of images, no global value bc that messes it up!
    curr_thresh = threshold_local(curr_denoised, block_size=35, method='gaussian', offset=0.05)
    next_thresh = threshold_local(next_denoised, block_size=35, method='gaussian', offset=0.05)
    
    #Otsu threshold --> see wiki page &skimage page for more details
    otsu_curr = threshold_otsu(curr_denoised)
    otsu_next = threshold_otsu(next_denoised)

This block **catches errors** incase the the output is **greater than 1.0**. Use the print check if your code gets stuck. You can throw this into google then to see what the issue is (most likley a nan error).

In [ ]:
    #catches errors, sometimes the value will be larger than 1.0, in that case this fixes that 
    curr_frame = (np.where(curr_denoised > otsu_curr, curr_float, 0.0) * 255).astype(np.uint8)
    next_frame = (np.where(next_denoised > otsu_next, next_float, 0.0) * 255).astype(np.uint8)

    print(f"Min: {next_float.min()}, Max: {next_float.max()}")
    #^comment this back in if code magically stops...

Now, we use this algorthim to create the **mask**. This mask is anywhere where the **Ostu** function is greater than the **denoised threshold** we created with the **gaussian**.

In [ ]:
    mask_layer = (curr_denoised <= otsu_curr)

Next, we set our **region size**. This is something you can change but do be warned that the smaller the region pixel size, the slower the code will run.

The code breaks down our enitre image into the pixel x pixel region sizes and compares them to the next frame. 

The winsize and search size should stay equal to oneanother. 

In [ ]:
    #how many pixels we want to evaluate at one time
    winsize = 16 
    searchsize = 16  
    overlap = 8 
    dt = 2.0 #time step, change depending on your data 

The following block is to determine our **speed changes**. PLEASE reference openPIV to understand what each of these things mean if an error arrises. 

In short, we are looking at the signal to noise changes in both of our frames using our pixel x pixel search size. We are also noting the change in time to calculate the speed change later. We are looking for changes in peaks between the two images within their respective search window. 

In [ ]:
u, v, sig2noise = pyprocess.extended_search_area_piv(
        curr_frame.astype(np.int16),
        next_frame.astype(np.int16),
        window_size=winsize,
        overlap=overlap,
        dt=dt,
        search_area_size=searchsize,
        sig2noise_method='peak2peak',
    )

We also record the **corrdiantes** using the same methodology. 

In [ ]:
 x, y = pyprocess.get_coordinates(
        image_size=curr_frame.shape,
        search_area_size=searchsize,
        overlap=overlap,
    )

This code catches any **crazy changes**. The threshold number will need to be modifed from data set to data set. **Note:** the compartment data will require a lower threshold number compared to the wave data. 

In [ ]:
 flags = validation.sig2noise_val(u, v, sig2noise, threshold=1.0015) #<-- change this if needed

Then we **generate the mask** to our search size for each image.

In [11]:
    y_indices = y.astype(int)
    x_indices = x.astype(int)
    
    grid_mask = mask_layer[y_indices, x_indices]

NameError: name 'y' is not defined

The following chunk of code is to **catch errors** such as Nan values. You might have to troubleshoot this from time to time if the library creates an update.

In [ ]:
u, v = filters.replace_outliers(
        u, v, flags, method='localmean', max_iter=10, kernel_size=2,
    )
    
    # Force float arrays so we can safely inject NaNs
    u = u.astype(float)
    v = v.astype(float)

    # HARD MASK FORCE: If the grid mask is True (background), set vector to NaN
    u[grid_mask] = np.nan
    v[grid_mask] = np.nan

Finally, we **apply the mask**. 

In [ ]:
GIGAMASK = grid_mask | np.isnan(u) | np.isnan(v)
    
    masked_u = np.ma.masked_array(u, mask=GIGAMASK)
    masked_v = np.ma.masked_array(v, mask=GIGAMASK)

We can caluclate the **magnitude** of the vector between frames; this is our speed output. We also append our bins with the current data along with index the time. 

In [ ]:
    #calculating the magnitiude of the vectors
    magnitude = np.ma.sqrt(masked_u**2 + masked_v**2)

    #Appending calculation metrics for current frame pair
    current_time = dt * i
    time_list.append(current_time)
    
    if np.any(~np.isnan(magnitude)) and magnitude.size > 0:
        avg_speed_list.append(np.nanmean(magnitude))
        max_speed_list.append(np.nanmax(magnitude))
        avg_move_x_list.append(np.nanmean(masked_u))
        avg_move_y_list.append(np.nanmean(masked_v))
    else:
    #If the frame has no movement or is completely masked, append 0 or np.nan
        avg_speed_list.append(0.0)
        max_speed_list.append(0.0)
        avg_move_x_list.append(0.0)
        avg_move_y_list.append(0.0)

The following has to do with the **presentation**. You can play with this to get a prettier output.  

It also save your figure and sitches all of the frames together. 

In [ ]:
 #PRESENTATION
    #start fresh after loop!
    ax.clear()
    #placing microscope image underneath 
    ax.imshow(curr_frame, alpha=0.5, cmap='gray')
    #draws in arrows
    Q = ax.quiver(
        x, 
        y, 
        masked_u, 
        -masked_v, 
        magnitude, 
        cmap="plasma",
        scale=115, 
        width=0.005,
    )
    Q.set_clim(vmin, vmax)
    #sacles colorbar incase or weirdness
    #cb.update_normal(Q)
    #more presentation stuff
    ax.invert_yaxis() 
    ax.set_xlim(0, first_img.shape[1])
    ax.set_ylim(first_img.shape[0], 0)
    ax.set_title(f"Velocity Field: {search_path } Minute: {3 * i}")
  
    frame_path = os.path.join(temp_dir, f"frame_{i:03d}.png")
    plt.savefig(frame_path, dpi=100)
    frame_files.append(frame_path)

The final chunk is more print checks and has to do with how you **name** your gifs and data files. **MAKE SURE YOU CHANGE THIS AFTER EACH FULL CODE RUN** otherwise you will overwrite your data. I did this a few times and it drove me nuts!

You can also assign where you would like the data and code saved.

In [ ]:
#print check!
print("\nAll individual vector frames rendered successfully.")
plt.close()
#stiching all of the new images together to form a gif
print("Stitching frames together...")
images = [iio.imread(f) for f in frame_files]
#lower duration for faster GIF
iio.imwrite("velocity_field_movie_mCherryPOS3algo-test13.gif", images, plugin="pillow", duration=250, loop=0)
import csv
csv_filename = "/Users/maggie/Desktop/localizedvelocity_analyticsmCherryPOS3algo-test13.csv"
#PRINT CHECK
print(f"Saving data to {csv_filename}...")
# putting the list we decalred before into rows
with open(csv_filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    # headers
    writer.writerow(["Time", "Avg_Speed", "Max", "AvgX", "AvgY"])
    writer.writerows(zip(time_list, avg_speed_list, max_speed_list, avg_move_x_list, avg_move_y_list))
#PRINT CHECK
print("CSV saved successfully!")
#housekeeping...not overloading computa with unneccesary files
print("Cleaning up temp files...")
for f in frame_files:
    os.remove(f)
os.rmdir(temp_dir)
#print check!
print("\nMovie saved successfully as 'velocity_field_movie_mCherryPOS3algo-test13.gif'!")

## END OF CODE!!!!

## ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
## PIV code WITH Manual Mask

### Same code as above BUT!

In between the: 

In [ ]:
flags = validation.sig2noise_val(u, v, sig2noise, threshold=1.0015)

AND 

In [ ]:
    y_indices = y.astype(int)
    x_indices = x.astype(int)

part of the code, you will put this: 

In [ ]:
 #manual masking, comment out when neccesary 
    #use sandbox to test coords
    img = np.zeros_like(curr_frame, dtype=bool)
    img2 = np.zeros_like(curr_frame, dtype=bool)
    img3 = np.zeros_like(curr_frame, dtype=bool)
    img4 = np.zeros_like(curr_frame, dtype=bool)
    
    rr, cc = polygon([0, 0, 1150, 1150], [0, 100, 100, 0],)
    
    qq, jj = polygon([0, 0, 450, 450], [0, 1150, 1150, 0],)

    yy, xx = polygon([0, 0, 1150, 1150], [730, 1150, 1150, 730],)
    
    mm, nn = polygon([780, 780, 1150, 1150], [0, 1150, 1150, 0],)
    
    img[rr, cc] = True
    img2[qq, jj] = True
    img3[yy, xx] = True  
    img4[mm, nn] = True 

Where the coordinates for rr,cc & qq, jj ect... will be changed. Each of these coordinates create a **polygon**. Where the first set of numbers in the brackets is your **y coords** and their corresponding **x coords**. 

The coords go in this order: **double click on me to reveal**
                                      1------2
                                      |      |
                                      |      |
                                      |      |
                                      4------3

Then in your **grid mask declaration**, you ened to **add** these **layers**. The creates, what I call, the **GIGAMASK**!

In [12]:
 grid_mask = (mask_layer[y_indices, x_indices] |
              img[y_indices, x_indices] | 
              img2[y_indices, x_indices] | 
              img3[y_indices, x_indices] | 
              img4[y_indices, x_indices])

NameError: name 'mask_layer' is not defined

### Note: you can commment this out of the code when you don't need it! Find the whole code on the GitHUB!

# End of PIV CODE!!!
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# Kymograph code:

## Again, we have a algomask version and a manual mask version.

### Starting with the algomask version: '

First, we start again with declaring our libraries and path.

In [ ]:
import csv
import os
from glob import glob
import matplotlib.pyplot as plt
import numpy as np
from skimage.io import imread
from skimage.util import img_as_float


search_path = search_path = "/Volumes/qiongy-data/Users/Maggie/111725 compartment data/POS 3, all cycles2/Cycle 12-1/POS 3 ratio-1*.tif"
image_paths = sorted(glob(search_path)) #note: you can cut of the number of frames you want to process: '[:-40]' for example

We also do a print check again to make sure we have the correct number of images to process: 

In [ ]:
if len(image_paths) < 2:
    raise ValueError("Double check your path configuration!")
print(f"Successfully tracked {len(image_paths)} ratio frames for mitotic wave analysis.")

We declare our time step and the number of pairs we will analyze.

Then, we must also convert our **height** and **width** of our images to B&W incase they are in color. Essentially 3D --> 2D

In [ ]:
dt = 2.0  # time step, minutes
total_pairs = len(image_paths) - 1

first_raw = imread(image_paths[0])
if first_raw.ndim == 3:
    first_img = img_as_float(first_raw[:, :, 0])
else:
    first_img = img_as_float(first_raw)
height, width = first_img.shape

Next, we can create some of our **bins** and a general 'shape' for our kymograph.

In [ ]:
kymo_x_wave_T = np.zeros((total_pairs, width))  
kymo_y_wave_T = np.zeros((total_pairs, height)) 

sum_diff = np.zeros((height, width), dtype=np.float64)
sum_diff_sq = np.zeros((height, width), dtype=np.float64)

time_list = []
global_activity = []

### Beginning of LOOP!

We start again by breaking apart the images into chunks of 10. This code runs quicker than PIV. We also update the time.

In [ ]:
print(f"Processing {total_pairs} frame pairs to capture dynamic wavefront shifts...")

for i in range(total_pairs):
    if (i + 1) % 10 == 0 or i == 0 or i == total_pairs - 1:
        print(f"Processing frame {i + 1} of {total_pairs}...")
    
    
    current_time = i * dt
    time_list.append(current_time)

Again, we need to declare our current and next frame for analysis. We also **convert** the color images to black and white, this time the actual images **data itself**, not just the height and width. 

In [ ]:
 current_time = i * dt
    time_list.append(current_time)
    
    raw_curr = imread(image_paths[i])
    raw_next = imread(image_paths[i + 1])
    
    if raw_curr.ndim == 3:
        curr_frame = img_as_float(raw_curr[:, :, 0])
        next_frame = img_as_float(raw_next[:, :, 0])
    else:
        curr_frame = img_as_float(raw_curr)
        next_frame = img_as_float(raw_next)

We use this chunk to set our scale and make sure to catch nan values. 

In [ ]:
    curr_frame = np.nan_to_num(curr_frame, nan=0.0, posinf=0.0, neginf=0.0)
    next_frame = np.nan_to_num(next_frame, nan=0.0, posinf=0.0, neginf=0.0)

    curr_frame = np.clip(curr_frame, 0.0, 2.0)
    next_frame = np.clip(next_frame, 0.0, 2.0) 

Then, we can caluclate the **absolute change in intensity** from frame to frame. 

In [ ]:
    diff_frame = np.abs(next_frame - curr_frame)

Then we can also see the median change on the **x** and **y** axis. We can break this abs change into components.  

In [ ]:
    kymo_x_wave_T[i, :] = np.median(diff_frame, axis=0) #declaring our axis
    kymo_y_wave_T[i, :] = np.median(diff_frame, axis=1) #declaring our axis

Our **global change** we declare as the **mean** of the abs change. We also calculate the sum of the differences and the sum of the differences sqaured for later calcs. 

In [ ]:
     global_activity.append(np.mean(diff_frame))

     sum_diff += diff_frame
     sum_diff_sq += diff_frame ** 2

### End of Loop!!!

We then **transpose** the data back into our expected output. space vs time. 

In [ ]:
kymo_x_wave = kymo_x_wave_T.T
kymo_y_wave = kymo_y_wave_T.T

Now, we can caluclate the **mean difference** across the whole data set. We can also caluclate the mean sqaured difference, the variance over our image(this is a work in progress, not neccesary for the kymo).

The variance map would be cool because it could tell us where some of the waves/activity typicall orginates but its not usually accurate (i would assume from qualitative understanding given the images themselves). 

In [ ]:
mean_diff = sum_diff / total_pairs
mean_diff_sq = sum_diff_sq / total_pairs


spatial_variance_map = mean_diff_sq - (mean_diff ** 2)
spatial_variance_map = np.clip(spatial_variance_map, 0, None)

Then we can find the **slope** of the changes at a given time. 

In [ ]:
with np.errstate(divide='ignore', invalid='ignore'):
    min_len = min(kymo_y_wave.shape[0], kymo_x_wave.shape[0])
    delta_ratio_kymo = kymo_y_wave[:min_len, :] / (kymo_x_wave[:min_len, :] + 1e-6)

Again, this is the spatial variance thingy... not super accurate but interesting. This chunk tries to find the area with the most variance since theroetically, this could be our point source. 

In [ ]:
wave_y_source, wave_x_source = np.unravel_index(np.argmax(spatial_variance_map), spatial_variance_map.shape)
print(f"\nCalculated Mitotic Wave Origin: X={wave_x_source}, Y={wave_y_source}")

Finally, the rest of the code is for saving the data and creating the figures. You can mess around with this if you would like! **AGAIN, make sure  TO CHANGE THE NAMES, OTHERWISE YOU WILL OVERWRITE YOUR DATA!!!!**

In [ ]:
#creating csv
csv_filename = "/Users/maggie/Desktop/wave_source_analyticsPOS3ratio-test13.csv"
print(f"Saving compiled spatial metrics to {csv_filename}...")
with open(csv_filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Time(min)", "GlobalWaveActivityMetric", "kymoxwave", "kymoywave"])
    writer.writerows(zip(time_list, global_activity, kymo_x_wave, kymo_y_wave))
    
#x-based kymograph 
plt.figure(figsize=(10, 4))
plt.imshow(kymo_x_wave, cmap='inferno', aspect='auto', extent=[0, (total_pairs * dt), 0, width])
plt.colorbar(label='Wave Activity ($\Delta$ Ratio)')
plt.xlabel('Time (minutes)')
plt.ylabel('X Position (pixels)')
plt.title('Mitotic Wave X-Kymograph (Median Projection)')
plt.tight_layout()
plt.savefig('wave_kymograph_xPOS3ratio-test13.png', dpi=150)
plt.show()

#slope-based kymograph
plt.figure(figsize=(10, 4))
plt.imshow(delta_ratio_kymo, cmap='twilight', aspect='auto', vmin=0, vmax=3, extent=[0, (total_pairs * dt), 0, min_len])
plt.colorbar(label='Propagation Ratio (Y/X)')
plt.xlabel('Time (minutes)')
plt.ylabel('Spatial Index')
plt.title('Wave Directional Dynamics: $\Delta$Y / $\Delta$X')
plt.tight_layout()
plt.savefig('wave_ratio_slopePOS3ratio-test13.png', dpi=150)
plt.show()

#wave source assumption map
#WORK IN PROGRESS...dont trust yet
plt.figure(figsize=(6, 6))
plt.imshow(first_img, cmap='gray')
#plt.scatter(wave_x_source, wave_y_source, )#marker='*', color='red')
plt.imshow(spatial_variance_map, cmap='jet', alpha=0.4)
plt.title('2D Wave Activity Map')
plt.colorbar(label='Temporal Variance')
plt.tight_layout()
plt.savefig('wave_source_origin_mapPOS3_ratio-test13.png', dpi=150)
plt.show()

#print showing the end of teh sim 
print("\nAnalysis complete! All files and plots are generated and saved.")

## End of CODE!!!

## ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

### Manual Masked Version:

You will use the same code above EXCEPT! you will put this function in right after your library calls and before your path information. You will need to change the following values depending on your image: 
- ymin
- ymax
- xmin
- xmax
- rr1, cc1
- rr2, cc2
- add more if you need but ensure that you declare another polymask and add that to your gigamask layer


Your ymin, ymax, xmin and xmax correspond to the pixel size of your cropped image and the polygons refine the cropped image even more. 

In [ ]:
#function declaration for processing the data and masking
#note that these y/x mins/maxs can be altered depending what you would like your window to look like
def process_and_mask_frame(raw_img, ymin=300, ymax=1100, xmin=0, xmax=400):
    
    #we must check if the incoming images are in B&W, otherwise we must convert them
    if raw_img.ndim == 3:
        img_float = img_as_float(raw_img[:, :, 0])
    else:
        img_float = img_as_float(raw_img)
    
    #decarling the img as a float in order to alter it
    img_float = np.nan_to_num(img_float, nan=0.0, posinf=0.0, neginf=0.0)
    img_float = np.clip(img_float, 0.0, 2.0)

    #a lot of this was taken directly from a tutorial
    #overall this chunk preforms a gaussian calculation to find the spot with the largest change and denoises the rest 
    img_denoised = gaussian(img_float, sigma=1.0)
    #this sets the threshold for our masking, anything below that point, we leave out i.e. darkness around the sample
    otsu_val = threshold_otsu(img_denoised)
    #and setting that threshold true for background noise
    intensity_mask = (img_denoised <= otsu_val) 
    
    #YOUR sandbox to play around with creating 'hard masks' and not auto ones
    #This should be used for specific isolation
    #google how to properly use the polygon function for more details
    #the more masks you need, the more mask decalrations you will need
    poly_mask1 = np.zeros_like(img_float, dtype=bool)
    poly_mask2 = np.zeros_like(img_float, dtype=bool)
    #poly_mask3 = np.zeros_like(img_float, dtype=bool)
    
    #parameters for the mask, these are like coordinates
    rr1, cc1 = polygon([0, 346, 346, 0],[0, 0, 1150, 1150])
    rr2, cc2 = polygon([0, 1150, 1150, 0],[330, 330, 1150, 1150])
    #rr3, cc3 = polygon([0, 1150, 1150, 0], [0, 0, 292, 292])
    
    #set those areas to true in order for them to block out analysis 
    poly_mask1[rr1, cc1] = True
    poly_mask2[rr2, cc2] = True
    #poly_mask3[rr3, cc3] = True
    
    #the whole mask = gigamask
    #composed of the algo and the hard masks 
    #using OR operator and opposed to &
    gigamask = intensity_mask | poly_mask1 | poly_mask2 #| poly_mask3
    
    #declaring our mask area as where the gigamask defines on our image
    masked_img = np.where(gigamask, 0.0, img_float)
    
    #returning that image cropped at our min/max values
    return masked_img[ymin:ymax, xmin:xmax]

The remainder of the code remains similar until right before the loop: 

In [ ]:
curr_frame_border = process_and_mask_frame(imread(image_paths[0]))

The updated loop looks like this: 

In [ ]:
#loop where caluclations come from
for i in range(total_pairs):
    
    #counting the time
    current_time = i * dt
    time_list.append(current_time)
    
    # Read ahead to next frame and process/mask it
    raw_next = imread(image_paths[i + 1])
    next_frame_border = process_and_mask_frame(raw_next)
    
    # Absolute difference calculation (now noise-free!)
    #this is where the global change data comes from 
    diff_frame = np.abs(next_frame_border - curr_frame_border)
    #add this to our array 
    all_diffs_2d.append(diff_frame)
    #taking that mean difference and adding it to our global activity array 
    global_activity.append(np.mean(diff_frame))
    
    #Setting up loop for the next frame v frame analysis 
    curr_frame_border = next_frame_border

#creating an array in the current array 
all_diffs_2d = np.array(all_diffs_2d)

#Generate Kymographs
# taking the median values (yeilded the clearest data, also at suggestion of lab)
#taking the axis = 1, x-axis and transposing the data 
kymo_x_wave = np.median(all_diffs_2d, axis=1).T 
#similar done here for y-axis 
kymo_y_wave = np.median(all_diffs_2d, axis=2).T  


#directional slope calc 
with np.errstate(divide='ignore', invalid='ignore'):
    min_len = min(kymo_y_wave.shape[0], kymo_x_wave.shape[0])
    delta_ratio_kymo = kymo_y_wave[:min_len, :] / (kymo_x_wave[:min_len, :] + 1e-6)

#Locate Wave Source Origin
#this is still a work in progress 
#maping the overall variance in the all_diffs_2d array
spatial_variance_map = np.var(all_diffs_2d, axis=0)
#declaring that teh osurce must come from the point of max change overtime
wave_y_source, wave_x_source = np.unravel_index(np.argmax(spatial_variance_map), spatial_variance_map.shape)

#We must adjust these variables if we are cropping the image, so do so here
ymin, xmin = 300, 0
global_wave_x = wave_x_source + xmin
global_wave_y = wave_y_source + ymin

All of the printing, save and calculating stuff is the same at the non-manual masked version. SEE the github for full code!!!

### End of CODE!!!

# End of image analysis code!!!